In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Ollama integrations
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings


# ----------------------------
# 1. LOAD LLM (OLLAMA)
# ----------------------------
# Make sure: `ollama serve` is running (usually automatic after install)
llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
    base_url="http://10.103.12.74:11434"
    # base_url="http://localhost:11434",  # uncomment if needed / remote
)

# ----------------------------
# 2. PROMPT (STRICT HISTORY)
# ----------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict knowledge assistant.\n"
     "You must ONLY answer from conversation history.\n"
     "If the answer is not present in history, say:\n"
     "\"I don't know from provided knowledge.\""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm

# ----------------------------
# 3. MESSAGE HISTORY STORE
# ----------------------------
store = {}

def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# ----------------------------
# 4. LOAD DATASET INTO HISTORY
# ----------------------------
csv_path = r"C:\Users\surya.adatravu\Documents\CONV_WINDOW_LLM_ANALYSIS\RA_FSM_QA.csv"
data_df = pd.read_csv(csv_path)

required_cols = {"Question", "Answer"}
if not required_cols.issubset(data_df.columns):
    raise ValueError(f"CSV must contain columns: Question, Answer. Found: {list(data_df.columns)}")

qa_pairs = list(zip(data_df["Question"].astype(str), data_df["Answer"].astype(str)))

SESSION_ID = "ra_fsm_session"
N = 100  # preload up to 100 Q/A pairs

history = get_history(SESSION_ID)
for q, a in qa_pairs[:N]:
    history.add_user_message(q)
    history.add_ai_message(a)

# ----------------------------
# 5. COSINE SIMILARITY CHECKER (OLLAMA EMBEDDINGS)
# ----------------------------
# Choose an embedding model you pulled with ollama (nomic-embed-text is a solid default)
embeddings = OllamaEmbeddings(model="nomic-embed-text",base_url="http://10.103.12.74:11434")

def similarity_score(text1, text2):
    v1 = embeddings.embed_query(text1)
    v2 = embeddings.embed_query(text2)
    return float(cosine_similarity([v1], [v2])[0][0])

def validate_answer(llm_answer, qa_pairs_to_check, threshold=0.82):
    best_score = -1.0
    best_answer = None
    for _, a in qa_pairs_to_check:
        s = similarity_score(llm_answer, a)
        if s > best_score:
            best_score = s
            best_answer = a
    return {"valid": best_score >= threshold, "score": best_score, "matched_answer": best_answer}

# ----------------------------
# 6. ASK PIPELINE
# ----------------------------
def ask(question, threshold=0.82):
    resp = chain_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": SESSION_ID}}
    )

    # ChatOllama returns an AIMessage
    llm_answer = resp.content

    validation = validate_answer(llm_answer, qa_pairs[:N], threshold=threshold)

    if not validation["valid"]:
        return {
            "question": question,
            "llm_answer": "Rejected: hallucination detected",
            "confidence": validation["score"],
            "closest_match": validation["matched_answer"]
        }

    return {
        "question": question,
        "llm_answer": llm_answer,
        "confidence": validation["score"],
        "closest_match": validation["matched_answer"]
    }

print(ask("provide details of all states in finate state machine "))
print(ask("provide details of confidence threshold levels"))
# print(ask("provide details of relavance threshold levels"))
print(ask("provide details of knowledge final answer prompts"))


{'question': 'provide details of all states in finate state machine ', 'llm_answer': 'Rejected: hallucination detected', 'confidence': 0.7123148215047302, 'closest_match': "It is routed to a 'Needs-Manual-Fix' state and added to the Missing-List."}
{'question': 'provide details of confidence threshold levels', 'llm_answer': 'Rejected: hallucination detected', 'confidence': 0.752715280961069, 'closest_match': 'Adapting confidence thresholds using expert feedback and adding domain transfer tests.'}
{'question': 'provide details of knowledge final answer prompts', 'llm_answer': 'Rejected: hallucination detected', 'confidence': 0.6906359900691686, 'closest_match': 'It assesses whether the system can answer a question confidently based on its internal context.'}
